# Lab 06-02 · Element Laws, Real Sources and Real Resistors

Companion to chapter [06-02](../../chapters/06-02-circuit-elements-and-ohms-law.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running. Parts 1 to 3 compare with ngspice when it is installed.

In [1]:
import math
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None

def ngspice_values(netlist):
    '''Run a netlist in batch mode; return every "name = value" it prints, as floats.'''
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        key, eq, rest = line.partition("=")
        if eq and rest.split():
            try:
                vals[key.strip().lower()] = float(rest.split()[0])
            except ValueError:
                pass
    return vals

print("ngspice available:", HAVE_NGSPICE)

ngspice available: True


## Part 1 — i–v characteristics

**🔮 Predict:** which of these pass through the origin: a 1 kΩ resistor, a 5 V source with 500 Ω internal resistance, a 2 mA source with 10 kΩ in parallel, a diode? What is the diode's slope $dv/di$ at 1 mA?

In [2]:
v = np.linspace(-1, 6, 701)
I_S, V_T = 1e-14, 0.025852
curves = {
    "1 kΩ resistor": v / 1e3,
    "5 V source, 500 Ω (delivered)": (5 - v) / 500,          # current delivered by the source
    "2 mA source ∥ 10 kΩ (delivered)": 2e-3 - v / 10e3,      # current delivered by the source
    "diode": I_S * np.expm1(np.clip(v, -1, 0.75) / V_T),
}
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, i in curves.items():
    ax.plot(v, i * 1e3, label=name)
ax.axhline(0, color="0.5", lw=0.8); ax.axvline(0, color="0.5", lw=0.8)
ax.set_ylim(-3, 12); ax.set_xlabel("v (V)"); ax.set_ylabel("i (mA)"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

vd = V_T * math.log(1e-3 / I_S + 1)
print(f"diode at 1 mA: v = {vd:.4f} V, static v/i = {vd / 1e-3:.1f} ohm, incremental dv/di = {V_T / (1e-3 + I_S):.2f} ohm")
if HAVE_NGSPICE:
    sim = ngspice_values("elements.cir")
    print(f"ngspice (300 K): v = {sim['vd1']:.4f} V, dv/di = {sim['rd1']:.2f} ohm")

diode at 1 mA: v = 0.6548 V, static v/i = 654.8 ohm, incremental dv/di = 25.85 ohm
ngspice (300 K): v = 0.6548 V, dv/di = 25.85 ohm


The two sources are plotted as the current they *deliver* (generator convention), so their lines slope downward; on the passive convention of the chapter's Figure 1 they would be mirrored. Only the resistor and the diode pass through the origin; the sources' lines cross the axes at their open-circuit voltage and short-circuit current. The resistor is linear, the diode is not: its static resistance at 1 mA is 654.8 Ω, its incremental resistance only 25.85 Ω, because the curve is steep there.

## Part 2 — A battery under load

**🔮 Predict:** as the load current rises from 0 to 5 A, what line does Example 06-02.3's cell (3.90 V, 0.40 Ω) follow?

In [3]:
i_load = np.linspace(0, 5, 21)
rng = np.random.default_rng(602)
# a bench measurement: the true cell (3.90 V, 0.40 ohm) read by a voltmeter with 3 mV of random error
v_meas = 3.90 - 0.40 * i_load + rng.normal(0, 3e-3, i_load.size)
slope, intercept = np.polyfit(i_load, v_meas, 1)
print(f"fitted from noisy readings: V_oc = {intercept:.3f} V, R_int = {-slope:.3f} ohm")
if HAVE_NGSPICE:
    print(f"ngspice sweep: R_int = {ngspice_values('elements.cir')['rint']:.3f} ohm")
assert abs(-slope - 0.40) < 0.005 and abs(intercept - 3.90) < 0.01

fitted from noisy readings: V_oc = 3.901 V, R_int = 0.400 ohm
ngspice sweep: R_int = 0.400 ohm


Two points determine the line; real measurements would scatter about it, and a least-squares fit (05-03) then estimates $R_{int}$ with an uncertainty.

## Part 3 — Cells in parallel

**🔮 Predict:** two cells, 4.0 V and 3.9 V, each 50 mΩ, are connected in parallel. What current flows between them?

In [4]:
dv = np.linspace(0, 0.6, 61)
i_circ = dv / (2 * 0.05)
heat = i_circ ** 2 * 2 * 0.05
print(f"0.1 V difference: {0.1 / 0.1:.1f} A, {0.1 ** 2 / 0.1:.2f} W of heat")
print(f"0.6 V difference: {0.6 / 0.1:.1f} A, {0.6 ** 2 / 0.1:.2f} W of heat")
if HAVE_NGSPICE:
    sim = ngspice_values("parallel_cells.cir")
    print(f"ngspice: 0.1 V -> {sim['i01']:.3f} A; Example 06-02.5 -> {sim['icirc']:.3f} A at {sim['vt']:.3f} V")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(dv, i_circ, label="circulating current (A)")
ax.plot(dv, heat, label="heat (W)")
ax.set_xlabel("difference in cell voltage (V)"); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

0.1 V difference: 1.0 A, 0.10 W of heat
0.6 V difference: 6.0 A, 3.60 W of heat
ngspice: 0.1 V -> 1.000 A; Example 06-02.5 -> 6.000 A at 3.900 V


Every 0.1 V of mismatch drives 1 A through the 0.1 Ω loop, and the heat grows as the square. Matched voltages, or isolating switches and diodes, are mandatory in packs.

## Part 4 — Where the E-series comes from

**🔮 Predict:** how many of the 24 standard E24 values does $10^{k/24}$, rounded to two significant figures, reproduce?

In [5]:
E12 = [1.0, 1.2, 1.5, 1.8, 2.2, 2.7, 3.3, 3.9, 4.7, 5.6, 6.8, 8.2]
E24 = [1.0, 1.1, 1.2, 1.3, 1.5, 1.6, 1.8, 2.0, 2.2, 2.4, 2.7, 3.0,
       3.3, 3.6, 3.9, 4.3, 4.7, 5.1, 5.6, 6.2, 6.8, 7.5, 8.2, 9.1]
for n, std in ((12, E12), (24, E24)):
    formula = [round(10 ** (k / n), 1) for k in range(n)]
    differ = [(f, s) for f, s in zip(formula, std) if abs(f - s) > 1e-9]
    print(f"E{n}: {len(differ)} of {n} differ (formula, standard): {differ}")
steps = [(E24[k + 1] if k + 1 < 24 else 10.0) / E24[k] for k in range(24)]
k_max = int(np.argmax(steps))
print(f"largest E24 step: {E24[k_max]} -> {E24[(k_max + 1) % 24]} (x{max(steps):.3f}); pure geometric step x{10 ** (1 / 24):.3f}")

E12: 5 of 12 differ (formula, standard): [(2.6, 2.7), (3.2, 3.3), (3.8, 3.9), (4.6, 4.7), (8.3, 8.2)]
E24: 8 of 24 differ (formula, standard): [(2.6, 2.7), (2.9, 3.0), (3.2, 3.3), (3.5, 3.6), (3.8, 3.9), (4.2, 4.3), (4.6, 4.7), (8.3, 8.2)]
largest E24 step: 1.3 -> 1.5 (x1.154); pure geometric step x1.101


Two-thirds of E24 follows the formula; the eight exceptions (2.7, 3.0, 3.3, 3.6, 3.9, 4.3, 4.7, 8.2) are historical values that predate the geometric definition and were kept when the series were standardized. The gaps in ±5% coverage (discussion question 3) have a different cause: rounding a geometric series to two significant figures ($10^{3/24} = 1.334 \to 1.3$, $10^{4/24} = 1.468 \to 1.5$) leaves the 1.3 → 1.5 step at ×1.154, so they would exist even without the historical exceptions.

## Part 5 — Tolerance, statistically

**🔮 Predict:** in 100 000 random builds of Example 06-02.4's LED circuit, will any LED current fall outside the worst-case range? What fraction falls within ±5% of nominal?

In [6]:
rng = np.random.default_rng(6)
N = 100_000
R = 300 * rng.uniform(0.95, 1.05, N)
Vs = 5.0 * rng.uniform(0.95, 1.05, N)
I = (Vs - 2.1) / R
lo, hi = (4.75 - 2.1) / 315, (5.25 - 2.1) / 285
nominal = 2.9 / 300
inside = np.mean(np.abs(I / nominal - 1) <= 0.05)
print(f"worst-case range {lo * 1e3:.3f} to {hi * 1e3:.3f} mA; Monte Carlo range {I.min() * 1e3:.3f} to {I.max() * 1e3:.3f} mA")
print(f"fraction within ±5% of nominal: {inside:.3f}")
assert lo <= I.min() and I.max() <= hi
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(I * 1e3, bins=80)
for x in (lo, hi):
    ax.axvline(x * 1e3, color="k", ls="--", lw=1)
ax.set_xlabel("LED current (mA)"); ax.set_ylabel("count"); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

worst-case range 8.413 to 11.053 mA; Monte Carlo range 8.416 to 11.046 mA
fraction within ±5% of nominal: 0.567


No sample can leave the worst-case range, by construction, and samples near its edges are rare: they need the supply and the resistor both near their limits. Just over half of the builds (57%) land within ±5% of nominal. Worst-case design guarantees function; statistics estimate how far from the edges a typical unit sits.

## Your turn

1. In Part 1, raise the diode current to 10 mA. How do the static and incremental resistances change?
2. In Part 3, connect the 4.0 V and 3.9 V cells to a 2 Ω load. Does the 3.9 V cell still absorb power?
3. In Part 5, replace the resistor with a ±1% part. How does the worst-case range change?

<details><summary>Answers</summary>

1. At 10 mA, $v = V_T\ln(10^{-2}/10^{-14} + 1)$ = 0.7143 V, so the static resistance is 71.43 Ω and the incremental resistance $V_T/I$ = 2.585 Ω: both fall about tenfold, the incremental one exactly in proportion to $1/I$.
2. KCL at the common node: $(4.0 - v)/0.05 + (3.9 - v)/0.05 = v/2$, so $v(20 + 20 + 0.5) = 80 + 78$ and $v = 158/40.5 = 3.901$ V. The 3.9 V cell's current out of its + terminal is $(3.9 - 3.901)/0.05 = -0.025$ A: it still absorbs a little (about 0.1 W) even with the load. The load's 1.95 A comes almost entirely from the stronger cell.
3. With ±1%: $I_{\max} = 3.15/297 = 10.61$ mA and $I_{\min} = 2.65/303 = 8.746$ mA. The range shrinks only from ±14% to about ±10%, because most of the spread comes from the ±5% supply acting on the 2.9 V difference. Tightening the part that does not dominate the error is wasted money; a current-source driver or a tighter supply would help more.

</details>